# Promotion Gate

## Notebook Contract
- **Objective:** answer one question — is this candidate model ready to ship? — by composing the modules already shipped in v0.4 (robust evaluation), v0.9 (governance), and v1.0 (artifact verification) into a single structured promotion report.
- **Inputs:** the synthetic support-ticket dataset + a small TF-IDF + LogReg baseline trained in the notebook; a candidate artifact directory built in a temp folder.
- **Outputs:** `promotion_report.md` (Markdown table + verdict) and `promotion_report.json` (machine-readable) under `reports/sample_run/promotion/`.
- **Expected runtime:** under 30 seconds on CPU.
- **Scope boundary:** every check reuses existing helpers; this notebook adds no statistical logic of its own.

## 1) Setup and Reproducibility

In [1]:
from __future__ import annotations

import json
import os
import platform
import random
import sys
import tempfile
from datetime import UTC, datetime
from pathlib import Path

ROOT = Path('..').resolve()
SRC_PATH = ROOT / 'src'
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

from hf_finetuning_lab.artifacts import verify_artifact
from hf_finetuning_lab.data.io import validate_text_classification_frame
from hf_finetuning_lab.evaluation.robust import (
    bootstrap_metric,
    expected_calibration_error,
    prediction_share_drift,
    subgroup_metrics,
)
from hf_finetuning_lab.experiments import hash_dataframe, make_run_id
from hf_finetuning_lab.governance import (
    DatasetCard,
    DatasetColumn,
    DatasetSplit,
    PromotionReport,
    ReproducibilityRecord,
    boolean_criterion,
    capture_environment,
    threshold_criterion,
    write_dataset_card,
    write_promotion_report,
    write_reproducibility_checklist,
    write_task_model_card,
)
from hf_finetuning_lab.sample_data import write_sample_data

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

print(f'Python: {sys.version.split()[0]}')
print(f'Platform: {platform.platform()}')
print(f"Timestamp (UTC): {datetime.now(UTC).isoformat(timespec='seconds')}")


Python: 3.12.11
Platform: Windows-11-10.0.26200-SP0
Timestamp (UTC): 2026-05-21T14:40:29+00:00


## 2) Promotion Thresholds

These are the only knobs the notebook exposes. In a real deployment pipeline they would live in a YAML/JSON file that travels with the project and gets version-controlled alongside the code.

In [2]:
DATA_PATH = ROOT / 'data' / 'raw' / 'support_tickets.csv'
PROMOTION_DIR = ROOT / 'reports' / 'sample_run' / 'promotion'
PROMOTION_DIR.mkdir(parents=True, exist_ok=True)

MIN_MACRO_F1_CI_LOW = 0.65
MAX_ECE = 0.20
MIN_SUBGROUP_RATIO = 0.70
MAX_TRAIN_TEST_PSI = 0.25
ROWS = 800
TEST_SIZE = 0.25
N_BOOTSTRAP = 300
POSITIVE_CLASS = 'billing'

thresholds_table = pd.DataFrame(
    [
        {'criterion': 'macro_f1_ci_low', 'direction': '>=', 'threshold': MIN_MACRO_F1_CI_LOW},
        {'criterion': 'expected_calibration_error', 'direction': '<=', 'threshold': MAX_ECE},
        {'criterion': 'subgroup_worst_over_best_f1', 'direction': '>=', 'threshold': MIN_SUBGROUP_RATIO},
        {'criterion': 'train_test_prediction_psi', 'direction': '<=', 'threshold': MAX_TRAIN_TEST_PSI},
    ]
)
thresholds_table

,criterion,direction,threshold
0,macro_f1_ci_low,>=,0.65
1,expected_calibration_error,<=,0.20
2,subgroup_worst_over_best_f1,>=,0.70
3,train_test_prediction_psi,<=,0.25


## 3) Train the Candidate

Same TF-IDF + LogReg baseline as the rest of the lab. We keep both the multiclass model and a binary `billing`-vs-other model so the calibration check has a clean positive-class probability.

In [3]:
DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
write_sample_data(output=DATA_PATH, rows=ROWS, seed=SEED)
df = pd.read_csv(DATA_PATH)
validate_text_classification_frame(df, text_col='text', label_col='label')
df['is_urgent'] = df['text'].str.contains('urgent', case=False, regex=False)
df['binary'] = (df['label'] == POSITIVE_CLASS).astype(int)
DATASET_HASH = hash_dataframe(df, columns=['text', 'label'])

train_df, test_df = train_test_split(df, test_size=TEST_SIZE, random_state=SEED, stratify=df['label'])


def make_pipeline() -> Pipeline:
    return Pipeline(
        steps=[
            ('tfidf', TfidfVectorizer(ngram_range=(1, 2), max_features=12000)),
            ('clf', LogisticRegression(max_iter=1000, random_state=SEED)),
        ]
    )


multiclass = make_pipeline().fit(train_df['text'], train_df['label'])
binary = make_pipeline().fit(train_df['text'], train_df['binary'])
multiclass_pred = multiclass.predict(test_df['text'])
binary_prob = binary.predict_proba(test_df['text'])[:, 1]

RUN_ID = make_run_id(prefix='promotion-candidate')
print(f'Run ID:       {RUN_ID}')
print(f'Dataset hash: {DATASET_HASH[:16]}\u2026')
print(f'Train rows: {len(train_df)} | test rows: {len(test_df)}')

Run ID:       promotion-candidate-20260521T144029Z-31fd38
Dataset hash: fafc0c83a222c6c5…
Train rows: 600 | test rows: 200


## 4) Stage the Candidate Artifact

Build a candidate model directory in a temp folder that satisfies the v1.0 artifact contract (`hf_finetuning_lab.artifacts.verify_artifact`). The governance helpers from notebook 07 (dataset card, task model card, reproducibility checklist) drop their outputs into that directory so the same checks pass without us hand-rolling files.

In [4]:
scratch = Path(tempfile.mkdtemp(prefix='promotion-candidate-'))
# Required + alternative-required slots are placeholders (verify_artifact only
# checks the layout, not the bytes).
(scratch / 'config.json').write_text('{}', encoding='utf-8')
(scratch / 'model.safetensors').write_text('placeholder', encoding='utf-8')
(scratch / 'tokenizer.json').write_text('{}', encoding='utf-8')
(scratch / 'tokenizer_config.json').write_text('{}', encoding='utf-8')
(scratch / 'special_tokens_map.json').write_text('{}', encoding='utf-8')

metrics_snapshot = {
    'macro_f1': float(f1_score(test_df['label'], multiclass_pred, average='macro', zero_division=0)),
    'weighted_f1': float(f1_score(test_df['label'], multiclass_pred, average='weighted', zero_division=0)),
}
(scratch / 'metrics.json').write_text(
    json.dumps(metrics_snapshot, indent=2), encoding='utf-8'
)

dataset_card = DatasetCard(
    name='synthetic-support-triage',
    description='Synthetic support-ticket data used for the promotion-gate demo.',
    task='text-classification',
    columns=[
        DatasetColumn(name='text', dtype='str', role='feature'),
        DatasetColumn(name='label', dtype='str', role='target'),
    ],
    splits=[
        DatasetSplit(name='train', n_rows=len(train_df)),
        DatasetSplit(name='test', n_rows=len(test_df)),
    ],
    limitations=['Synthetic labels; not a real operational taxonomy.'],
)
write_dataset_card(dataset_card, scratch / 'dataset_card.md')
write_task_model_card(
    output_path=scratch / 'model_card.md',
    model_name='tfidf-logreg-promotion-candidate',
    task='text-classification',
    label_names=sorted(test_df['label'].unique().tolist()),
    metrics=metrics_snapshot,
    extra_limitations=['Synthetic dataset; metrics are illustrative.'],
)
environment = capture_environment(cwd=ROOT)
write_reproducibility_checklist(
    ReproducibilityRecord(
        run_id=RUN_ID,
        task='text-classification',
        seed=SEED,
        dataset_hash=DATASET_HASH,
        model_name='tfidf-logreg-promotion-candidate',
        config={'rows': ROWS, 'test_size': TEST_SIZE},
        metrics=metrics_snapshot,
        environment=environment,
        notes='Promotion-gate demo run.',
    ),
    scratch / 'reproducibility.md',
)
print(f'Candidate artifact at: {scratch}')
sorted(p.name for p in scratch.iterdir())


Candidate artifact at: C:\Users\diogo\AppData\Local\Temp\promotion-candidate-fjv2y3vi


['config.json',
 'dataset_card.md',
 'metrics.json',
 'model.safetensors',
 'model_card.md',
 'reproducibility.json',
 'reproducibility.md',
 'special_tokens_map.json',
 'tokenizer.json',
 'tokenizer_config.json']

## 5) Run the Checks

Each block builds a `PromotionCriterion` via either `threshold_criterion` (numeric) or `boolean_criterion` (binary). We collect them in order and feed them to `PromotionReport`.

In [5]:
artifact_report = verify_artifact(scratch)
artifact_criterion = boolean_criterion(
    name='artifact_contract',
    ok=artifact_report.ok and not artifact_report.warnings,
    detail=(
        f'ok={artifact_report.ok} | warnings={len(artifact_report.warnings)} | '
        f'missing={len(artifact_report.missing)}'
    ),
)
artifact_criterion

PromotionCriterion(name='artifact_contract', status='pass', detail='ok=True | warnings=0 | missing=0', value=None, threshold=None)

In [6]:
boot = bootstrap_metric(
    y_true=test_df['label'].to_numpy(),
    y_pred=multiclass_pred,
    metric_fn=lambda yt, yp: float(f1_score(yt, yp, average='macro', zero_division=0)),
    n_iter=N_BOOTSTRAP,
    seed=SEED,
)
macro_f1_criterion = threshold_criterion(
    name='macro_f1_ci_low',
    value=boot['ci_low'],
    threshold=MIN_MACRO_F1_CI_LOW,
    direction='ge',
    detail_unit=' macro F1 (95% bootstrap CI low)',
)
{'point': boot['value'], 'ci_low': boot['ci_low'], 'ci_high': boot['ci_high']}

{'point': 1.0, 'ci_low': 1.0, 'ci_high': 1.0}

In [7]:
ece = expected_calibration_error(test_df['binary'].to_numpy(), binary_prob, n_bins=10)
ece_criterion = threshold_criterion(
    name='expected_calibration_error',
    value=ece,
    threshold=MAX_ECE,
    direction='le',
    detail_unit=' ECE (binary view)',
)
ece

0.07487215744509829

In [8]:
subgroup_table = subgroup_metrics(
    test_df['label'].to_numpy(),
    multiclass_pred,
    groups=test_df['is_urgent'].to_numpy(),
    metric_fns={
        'macro_f1': lambda yt, yp: float(f1_score(yt, yp, average='macro', zero_division=0)),
    },
)
f1_values = subgroup_table['macro_f1']
subgroup_ratio = float(f1_values.min() / f1_values.max()) if f1_values.max() > 0 else 0.0
subgroup_criterion = threshold_criterion(
    name='subgroup_worst_over_best_f1',
    value=subgroup_ratio,
    threshold=MIN_SUBGROUP_RATIO,
    direction='ge',
    detail_unit=' worst/best macro F1 across is_urgent groups',
)
subgroup_table.assign(group=['urgent' if g else 'non-urgent' for g in subgroup_table.index]).set_index('group').round(4)

,count,macro_f1
group,,
non-urgent,177,1.0
urgent,23,1.0


In [9]:
train_pred = multiclass.predict(train_df['text'])
drift = prediction_share_drift(train_pred, multiclass_pred)
psi_total = float(drift.attrs['psi_total'])
psi_criterion = threshold_criterion(
    name='train_test_prediction_psi',
    value=psi_total,
    threshold=MAX_TRAIN_TEST_PSI,
    direction='le',
    detail_unit=' PSI (train vs test predictions)',
)
drift.round(4)

,label,share_a,share_b,delta,psi
0,account,0.1900,0.190,0.0000,0.0000
1,billing,0.1700,0.170,0.0000,0.0000
2,cancellation,0.1217,0.125,0.0033,0.0001
3,delivery,0.1183,0.115,-0.0033,0.0001
4,security,0.1933,0.195,0.0017,0.0000
5,technical,0.2067,0.205,-0.0017,0.0000


In [10]:
governance_files = {
    name: (scratch / name).is_file()
    for name in ('dataset_card.md', 'model_card.md', 'reproducibility.md')
}
governance_criterion = boolean_criterion(
    name='governance_artifacts_present',
    ok=all(governance_files.values()),
    detail=' | '.join(f'{k}={v}' for k, v in governance_files.items()),
)
governance_criterion

PromotionCriterion(name='governance_artifacts_present', status='pass', detail='dataset_card.md=True | model_card.md=True | reproducibility.md=True', value=None, threshold=None)

In [11]:
env = environment
git_clean = env.get('git_dirty') is False
repro_criterion = boolean_criterion(
    name='git_working_tree_clean',
    ok=bool(git_clean),
    detail=(
        f"git_commit={env.get('git_commit')!s} | git_dirty={env.get('git_dirty')!r}"
    ),
)
env

{'python_version': '3.12.11',
 'platform': 'Windows-11-10.0.26200-SP0',
 'captured_at_utc': '2026-05-21T14:40:29+00:00',
 'git_commit': '224ec06',
 'git_dirty': True}

## 6) Build and Persist the Promotion Report

In [12]:
report = PromotionReport(
    run_id=RUN_ID,
    model_name='tfidf-logreg-promotion-candidate',
    criteria=[
        artifact_criterion,
        macro_f1_criterion,
        ece_criterion,
        subgroup_criterion,
        psi_criterion,
        governance_criterion,
        repro_criterion,
    ],
    notes='Generated by notebooks/10_promotion_gate.ipynb.',
)
report_path = write_promotion_report(report, PROMOTION_DIR / 'promotion_report.md')
print(f'Verdict:  {"PROMOTE" if report.should_promote else "BLOCK"}')
print(f'Failed:   {len(report.failed)}')
print(f'Passed:   {len(report.passed)}')
print(f'Skipped:  {len(report.skipped)}')
print(f'Written:  {report_path}')
print(f'JSON:     {report_path.with_suffix(".json")}')
print()
print(report_path.read_text(encoding='utf-8'))

Verdict:  BLOCK
Failed:   1
Passed:   6
Skipped:  0
Written:  C:\Users\diogo\work_code\portfolio\huggingface-finetuning-lab\reports\sample_run\promotion\promotion_report.md
JSON:     C:\Users\diogo\work_code\portfolio\huggingface-finetuning-lab\reports\sample_run\promotion\promotion_report.json

# Promotion Report: promotion-candidate-20260521T144029Z-31fd38

## Verdict

**BLOCK** — `should_promote = False`.

- **Model:** `tfidf-logreg-promotion-candidate`
- **Generated (UTC):** `2026-05-21T14:40:30+00:00`
- **Failed:** 1 | **Passed:** 6 | **Skipped:** 0

## Criteria

| status | criterion | value | threshold | detail |
| --- | --- | --- | --- | --- |
| PASS | `artifact_contract` | — | — | ok=True | warnings=0 | missing=0 |
| PASS | `macro_f1_ci_low` | 1.0 | 0.65 | 1.0000 >= 0.6500 macro F1 (95% bootstrap CI low) |
| PASS | `expected_calibration_error` | 0.07487215744509829 | 0.2 | 0.0749 <= 0.2000 ECE (binary view) |
| PASS | `subgroup_worst_over_best_f1` | 1.0 | 0.7 | 1.0000 >= 0.7000

In [13]:
pd.DataFrame(
    [
        {
            'status': c.status.upper(),
            'name': c.name,
            'value': c.value,
            'threshold': c.threshold,
            'detail': c.detail,
        }
        for c in report.criteria
    ]
)

,status,name,value,threshold,detail
0,PASS,artifact_contract,NaN,NaN,ok=True | warnings=0 | missing=0
1,PASS,macro_f1_ci_low,1.000000,0.65,1.0000 >= 0.6500 macro F1 (95% bootstrap CI low)
2,PASS,expected_calibration_error,0.074872,0.20,0.0749 <= 0.2000 ECE (binary view)
3,PASS,subgroup_worst_over_best_f1,1.000000,0.70,1.0000 >= 0.7000 worst/best macro F1 across is...
4,PASS,train_test_prediction_psi,0.000213,0.25,0.0002 <= 0.2500 PSI (train vs test predictions)
5,PASS,governance_artifacts_present,NaN,NaN,dataset_card.md=True | model_card.md=True | re...
6,FAIL,git_working_tree_clean,NaN,NaN,git_commit=224ec06 | git_dirty=True


## 7) Checklist
- One report file per promotion decision: `promotion_report.md` (human-readable verdict + criteria table) plus `promotion_report.json` (machine-readable for CI gates).
- `should_promote` is `True` only when zero criteria fail; skipped criteria do not block (e.g. GPU smoke when no GPU is available).
- The notebook is the *composition*; the checks live in `evaluation.robust` (CI / ECE / subgroup / drift), `artifacts` (layout contract), and `governance` (dataset card + model card + reproducibility checklist).
- For production: load the thresholds from a versioned config file, run this notebook as a script in the deployment pipeline, and gate the promotion step on `should_promote`.